# Marker Repo - submit lists

## Loading packages

In [2]:
import markerrepo.marker_repo as mr
import markerrepo.update_uids as u_uids
import markerrepo.generate_metafile as gm
import markerrepo.validate_yaml as validate
import markerrepo.utils as utils

## Settings

In [3]:
# The path of the repository
REPO_PATH = "/mnt/workspace/mkessle/projects/annotate_by_marker_and_features"
# The path where the lists of the Marker Repo are stored - probable 'REPO_PATH/lists'
REPO_LISTS_PATH = "/mnt/workspace/mkessle/projects/annotate_by_marker_and_features/lists"
# The path of the list to be added to the marker repo.
LIST_PATH = "/mnt/workspace/mkessle/projects/sc_framework/sctoolbox/data/gene_lists/rat_cellcycle_genes.txt"

# The column where the markers are stored
marker_col = 0
# The column where the information of the markers are stored (e.g. cell type, phase, ...).
# If there are only markers available (one column) enter a name, which describes the markers (e.g. gender genes).
info_col = 1

## Get whitelists

Pull whitelist repository and update if necessary.

In [4]:
mr.get_whitelists(REPO_PATH)

Fetching whitelists...

Done!


Get essential metadata: Liste name, organism and marker type

In [ ]:
LIST_NAME = input("Please enter the name of the marker list: ")
ORGANISM = mr.select(key="organism")
MARKER_TYPE = mr.select(key="marker_type")

Read genes whitelist in order to filter and extend marker genes

In [ ]:
if MARKER_TYPE == "Genes":
    gene_dict = mr.get_gene_dict(ORGANISM)

## Read, filter, extend and transform marker list

Read marker list

In [ ]:
markers = mr.get_list(LIST_PATH, info_col=info_col, marker_col=marker_col)
markers['Marker'] = markers['Marker'].str.upper()
display(markers)

Filter marker list

In [ ]:
# TODO - select filter: keep protein conding only? keep noncoding? keep ... ?
if MARKER_TYPE == "Genes":
    markers_filtered = markers[markers['Marker'].isin(gene_dict.keys())]
    display(markers_filtered)
    markers_filtered = markers_filtered.drop_duplicates()
    display(markers_filtered)

Extend marker list

In [ ]:
if MARKER_TYPE == "Genes":
    markers_extended = mr.update_markers(markers_filtered, gene_dict)
    display(markers_extended)

Convert marker list in order to append it to the yaml file

In [ ]:
if MARKER_TYPE == "Genes":
    marker_dict = mr.dataframe_to_dict(markers_extended)
else:
    # TODO - filter genomic regions?
    marker_dict = mr.dataframe_to_dict(markers)
    
marker_list = []
for name in marker_dict.keys():
    marker_list.append({'name': name, 'markers': marker_dict[name]})

## Enter metadata

Enter general metadata, tags and add marker list(s) automatically.

In [ ]:
# TODO - solve problem of whitelist autocompletion (tissue)

# Get UID
UID = mr.get_uid(REPO_LISTS_PATH)
file_name = f"{LIST_NAME}_{UID}.yaml"

# Enter metadata
gm.generate_file(REPO_LISTS_PATH, UID, LIST_NAME, False, marker_list, ORGANISM, MARKER_TYPE)

## Validation

In [ ]:
if validate.validate_file(utils.read_in_yaml(f"{REPO_LISTS_PATH}/{file_name}")):
    print(f"No errors were found concerning the '{LIST_NAME}' marker list.")

## Push list to repository

In [ ]:
mr.push_marker_list(REPO_PATH, f"{REPO_LISTS_PATH}/{file_name}")